# IntensityScan2 — figures

Interactive version of `generate_figures.py`.
Run cells in order; figures are displayed inline instead of saved to PDF.

In [ ]:
%matplotlib inline
import gzip, json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path().resolve().parents[1]          # sps-momentum-acceptance root
THIS_DIR  = Path().resolve()                     # studies/intensity_scan2

sys.path.insert(0, str(REPO_ROOT / "helper_functions"))
sys.path.insert(0, str(REPO_ROOT / "helper_functions" / "intensity_helpers"))
sys.path.insert(0, str(REPO_ROOT / "helper_functions" / "tune_diagram_helpers"))
sys.path.insert(0, str(THIS_DIR))

from load_paths import get_path
import acceptance_centers as ac
import midpoints_analysis as mpa
from intensity_loss import plot_intensity_drop
from generate_intensityscan2_midpoint_qx_plots import (
    build_tune_midpoints_from_maps,
    plot_qx_midpoints_vs_xi,
    plot_qy_midpoints_vs_xi,
)
from plot_midpoints_centres_acceptance import generate as _generate_midpoints_set
from generate_figures import (
    load_study_results,
    load_md_midpoints,
    STUDY_RESULTS,
    MD_ROOT,
    MAP_LINEAR,
    MAP_ERRORS,
    SWEEP_PER_TURN,
    NUM_PARTICLES,
    FONTSIZE,
    CHROMAS_COMPARISON,
    COLORS_COMPARISON,
)

## Load data

In [ ]:
data = load_study_results(STUDY_RESULTS)
md   = load_md_midpoints(MD_ROOT)

chromas = sorted(data["linear"].keys())
print(f"Chromaticity values: {chromas}")
print(f"MD midpoints loaded: {md is not None}")

## Figure 1 — Intensity loss vs δ (no-errors model)

Shows the normalised intensity as a function of momentum offset δ for each chromaticity value, using the linear (no-errors) machine model.

In [ ]:
fig, _ = plot_intensity_drop(data, line_types=["linear"], fontsize=FONTSIZE, legend=False)
plt.show()

## Figure 2 — Error-model comparison

Selected chromaticity values (ξ = 0.5, 0.7, 1.0): solid = no errors, dashed = with errors.

In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(8, 6))
for chroma in CHROMAS_COMPARISON:
    for line_type, ls in [("linear", "-"), ("errors", "--")]:
        d = data[line_type].get(chroma)
        if d is None:
            continue
        for plane, curve in d.items():
            ax.plot(curve["deltas"], curve["values"],
                    color=COLORS_COMPARISON[chroma], linestyle=ls, linewidth=1.5)

chroma_handles = [Line2D([0],[0], color=COLORS_COMPARISON[c], lw=2, label=rf"$\xi={c}$") for c in CHROMAS_COMPARISON]
model_handles  = [Line2D([0],[0], color="black", lw=2, ls="-",  label="No errors"),
                  Line2D([0],[0], color="black", lw=2, ls="--", label="With errors")]
ax.legend(handles=chroma_handles + model_handles, fontsize=FONTSIZE-2, frameon=True)
ax.set_xlabel(r"$\delta$", fontsize=FONTSIZE)
ax.set_ylabel("Normalised Intensity", fontsize=FONTSIZE)
ax.tick_params(labelsize=FONTSIZE-2)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Figures 3–12 — Midpoints, centres, acceptance, Q at δ₅₀

Two sets (no-errors only / both models), five figures each:
- **midpoints**: δ₅₀ vs ξ
- **centres**: (δ⁺₅₀ + δ⁻₅₀)/2 vs ξ  
- **acceptance**: (δ⁺₅₀ − δ⁻₅₀) vs ξ  
- **qx50**: Qx at δ₅₀ (from tune maps) vs ξ  
- **qy50**: Qy at δ₅₀ (from tune maps) vs ξ

These call `plot_midpoints_centres_acceptance.generate()` which saves PDFs to Figures/ and also displays internally. To see them inline here, call the underlying helper functions directly (see cells below).

In [ ]:
COLOURS = {"linear": "royalblue", "errors": "crimson", "MD": "blueviolet"}

midpoints = mpa.get_midpoints(data)

# --- midpoints (δ₅₀ vs ξ) ---
fig, ax = mpa.plot_midpoints(midpoints, midpoints_md=md, line_types=["linear", "errors"],
                              colours=COLOURS, fontsize=FONTSIZE)
ax.set_title(r"$\delta_{50\%}$ vs chromaticity")
plt.show()

# --- centres ---
fig, ax = ac.plot_centers(midpoints, line_types=["linear", "errors"],
                           colours=COLOURS, md_midpoints=md, fontsize=FONTSIZE)
plt.show()

# --- acceptance ---
fig, ax = ac.plot_acceptance(midpoints, line_types=["linear", "errors"],
                              colours=COLOURS, md_midpoints=md, fontsize=FONTSIZE)
ax.set_ylim(bottom=0)
plt.show()

In [ ]:
# --- Qx and Qy at δ₅₀ from tune maps ---
from pathlib import Path as _P
import tempfile

qx_mid, _ = build_tune_midpoints_from_maps(
    midpoints, MAP_LINEAR,
    line_types=["linear", "errors"], map_axis="x",
    fixed_map_xi=0.0, coupled_map_xi=True, component="qx",
    map_root_errors=MAP_ERRORS,
)
qy_mid, _ = build_tune_midpoints_from_maps(
    midpoints, MAP_LINEAR,
    line_types=["linear", "errors"], map_axis="x",
    fixed_map_xi=0.0, coupled_map_xi=True, component="qy",
    map_root_errors=MAP_ERRORS,
)

# plot_qx/qy_midpoints_vs_xi save to a path — use a temp file and show inline
import tempfile, os
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
    tmp_path = _P(tmp.name)

plot_qx_midpoints_vs_xi(qx_mid, tmp_path, line_types=["linear", "errors"],
                         qx_md_midpoints=None, fontsize=FONTSIZE)
from IPython.display import Image, display
display(Image(tmp_path))

plot_qy_midpoints_vs_xi(qy_mid, tmp_path, line_types=["linear", "errors"],
                         qy_md_midpoints=None, fontsize=FONTSIZE)
display(Image(tmp_path))
os.unlink(tmp_path)

## Figure 13 — Selected coupled-ξ tune diagram (WithErrors ChromaScanX)

Overlays trajectories for ξ = −1.5, −0.5, 0, 0.5, 1.5 on the tune diagram.
Requires the corresponding maps in `sps-chromaticity-maps/with_errors/`.

In [ ]:
from plot_selected_coupled_chromascan import main as _run_chroma
try:
    _run_chroma()   # saves PDF to sps-chromaticity-maps/with_errors/
    print("Saved — open the PDF in sps-chromaticity-maps/with_errors/")
except FileNotFoundError as exc:
    print(f"[skip] {exc}")